# Limpieza y Validación de Sabanas Modelo Yastas → Parquet por mes


1. **Carga** cada pestaña/mes disponible.
2. **Limpia**: normaliza el ID del comercio (blindado contra el bug de `10002` vs `10002.0`), convierte a numérico las columnas del modelo, corrige nulos y valores imposibles (negativos en conteos/montos, targets fuera de `{0,1}`).
3. **Valida**: genera un reporte de calidad por mes — cuántas filas se vieron afectadas por cada problema, y qué columnas del modelo faltan en cada hoja.
4. **Exporta**: un archivo `.parquet` por mes, con **únicamente** las columnas que tu modelo usa (ID + las 32 variables de Fraccionamiento/Nocturna/CdC/BE + los 5 targets) — nada de columnas descriptivas que el modelo no toca.


In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 120)

## 1. Configuración

In [ ]:
# ── Rutas ────────────────────────────────────────────────────────────────────
RUTA_EXCEL = Path(
    r"D:\Users\lumonroy\OneDrive - Compartamos Banco\Escritorio\Yastas Modelo\DatosModelo.xlsx"
)
CARPETA_SALIDA = Path(
    r"D:\Users\lumonroy\OneDrive - Compartamos Banco\Escritorio\Yastas Modelo\Parquet Modelo Yastas"
)
CARPETA_SALIDA.mkdir(parents=True, exist_ok=True)

# ── Nombres posibles de la columna de ID ───────────
COLUMNA_ID_POSIBLES = ["ID COMERCIO aux", "ID COB", "ID CMR aux"]

# ── Targets ────────────────────────
TARGETS = [
    "Tuvo_Alertas_Malas_Practicas", "Tuvo_Alertas_BE", "Tuvo_Alertas_Fracc",
    "Tuvo_Alertas_Noche", "Tuve_Alertas_CDC",
]

# ── Variables que usa el modelo, agrupadas igual que en el entrenamiento ─────
VARS_BASE = ["Transacciones", "Monto operado"]
VARS_BE = ["Total Alertas BE", "Txs con BE", "Monto Operado con BE"]
VARS_FRACC = [
    "txs_borde_umbral", "velocity_max_1h", "entropy_monto", "ref_unicas_ratio",
    "txs_ref_repetida", "pct_txs_borde", "coef_variacion_monto",
    "ratio_borde", "ratio_velocidad", "ratio_rafaga", "ratio_mismo_monto",
]
VARS_NOCHE = [
    "Txs 11pm a 6 am", "Monto Operado de 11pm a 6 am", "Txs de CB de 11pm a 6 am",
    "pct_monto_nocturno", "monto_noc_vs_propio_diurno", "tiempo_inactividad_diurna_hrs",
    "txs_noc_dias_distintos", "Txs Fracc de 11pm a 6am",
]
VARS_CDC = [
    "Txs Cob de Cobs", "A cuantos comisionistas invierte", "cuentas_destino_distintas",
    "cuentas_origen_distintas", "monto_entrante_cdc", "ratio_circularidad_monto",
    "flag_ciclo_24h", "tiempo_retorno_hrs_p50",
]

# Union sin duplicados, en el mismo orden en que se usan al entrenar
VARS_MODELO = list(dict.fromkeys(VARS_BASE + VARS_BE + VARS_FRACC + VARS_NOCHE + VARS_CDC))

# ── Columnas donde un valor negativo es imposible (para la validación) ───────
COLUMNAS_CONTEO = [
    "Transacciones", "Total Alertas BE", "Txs con BE", "txs_borde_umbral",
    "velocity_max_1h", "txs_ref_repetida", "Txs 11pm a 6 am",
    "Txs de CB de 11pm a 6 am", "txs_noc_dias_distintos", "Txs Fracc de 11pm a 6am",
    "Txs Cob de Cobs", "A cuantos comisionistas invierte",
    "cuentas_destino_distintas", "cuentas_origen_distintas",
]
COLUMNAS_MONTO = [
    "Monto operado", "Monto Operado con BE", "Monto Operado de 11pm a 6 am",
    "monto_entrante_cdc",
]

print(f"Variables del modelo a conservar: {len(VARS_MODELO)}")
print(f"Targets a conservar: {len(TARGETS)}")

## 2. Funciones de limpieza

### 2.1 Normalización del ID


In [ ]:
def limpiar_id(serie: pd.Series) -> pd.Series:
    s = serie.astype(str).str.strip()
    s = s.str.replace(r"\.0+$", "", regex=True)
    return s


def encontrar_columna_id(columnas) -> str | None:
    for c in COLUMNA_ID_POSIBLES:
        if c in columnas:
            return c
    return None

### 2.2 Carga, limpieza y validación de una hoja.

In [ ]:
def cargar_y_limpiar_hoja(ruta_excel: Path, hoja: str) -> tuple[pd.DataFrame | None, dict]:
    reporte = {"hoja": hoja, "advertencias": [], "errores": [],
               "n_filas_original": 0, "n_filas_final": 0,
               "vars_presentes": 0, "targets_presentes": 0}

    df = pd.read_excel(ruta_excel, sheet_name=hoja)
    df.columns = df.columns.str.strip()
    reporte["n_filas_original"] = len(df)

    # ── ID ───────────────────────────────────────────────────────────────
    col_id = encontrar_columna_id(df.columns)
    if col_id is None:
        reporte["errores"].append(
            f"No se encontró columna de ID (probé {COLUMNA_ID_POSIBLES}); se omite la hoja."
        )
        return None, reporte

    df = df.rename(columns={col_id: "ID COB"})
    df["ID COB"] = limpiar_id(df["ID COB"])

    n_sin_id = df["ID COB"].isin(["", "nan", "none", "0"]).sum()
    if n_sin_id:
        reporte["advertencias"].append(f"{n_sin_id} filas sin ID válido (se eliminan)")
    df = df[~df["ID COB"].isin(["", "nan", "none", "0"])]

    n_dup = df.duplicated(subset="ID COB", keep="last").sum()
    if n_dup:
        reporte["advertencias"].append(
            f"{n_dup} IDs duplicados (se conserva la última fila de cada uno)"
        )
    df = df.drop_duplicates(subset="ID COB", keep="last")

    # ── Qué columnas del modelo sí están en esta hoja ───────────────────────
    vars_presentes = [c for c in VARS_MODELO if c in df.columns]
    vars_faltantes = [c for c in VARS_MODELO if c not in df.columns]
    if vars_faltantes:
        reporte["advertencias"].append(
            f"Variables del modelo NO encontradas en esta hoja: {vars_faltantes}"
        )

    targets_presentes = [c for c in TARGETS if c in df.columns]
    targets_faltantes = [c for c in TARGETS if c not in df.columns]
    if targets_faltantes:
        reporte["advertencias"].append(f"Targets no encontrados en esta hoja: {targets_faltantes}")

    columnas_a_limpiar = vars_presentes + targets_presentes

    # ── Forzar numérico, reportando lo que no se pudo convertir ─────────────
    for c in columnas_a_limpiar:
        antes = pd.to_numeric(df[c], errors="coerce")
        n_no_numerico = antes.isna().sum() - df[c].isna().sum()
        if n_no_numerico > 0:
            reporte["advertencias"].append(
                f"'{c}': {n_no_numerico} valores no numéricos (se vuelven nulos)"
            )
        df[c] = antes

    # ── Nulos: se rellenan con 0, pero SIEMPRE se reporta cuántos y en qué columna ──
    for c in columnas_a_limpiar:
        n_nulos = int(df[c].isna().sum())
        if n_nulos > 0:
            pct = n_nulos / len(df) * 100
            reporte["advertencias"].append(f"'{c}': {n_nulos} nulos ({pct:.1f}%) -> se rellenan con 0")
            df[c] = df[c].fillna(0)

    # ── Negativos imposibles (conteos y montos no pueden ser negativos) ─────
    for c in COLUMNAS_CONTEO + COLUMNAS_MONTO:
        if c in df.columns:
            n_neg = int((df[c] < 0).sum())
            if n_neg:
                reporte["advertencias"].append(
                    f"'{c}': {n_neg} valores negativos -> se corrigen a 0"
                )
                df.loc[df[c] < 0, c] = 0

    # ── Targets: deben ser estrictamente 0/1; lo que no, se descarta (no se adivina) ──
    for c in targets_presentes:
        vals_raros = ~df[c].isin([0, 1])
        n_raros = int(vals_raros.sum())
        if n_raros:
            ejemplos = df.loc[vals_raros, c].unique()[:5].tolist()
            reporte["errores"].append(
                f"'{c}': {n_raros} valores fuera de {{0,1}} (ej: {ejemplos}) -> esas filas se descartan"
            )
            df = df[~vals_raros]

    reporte["n_filas_final"] = len(df)
    reporte["vars_presentes"] = len(vars_presentes)
    reporte["targets_presentes"] = len(targets_presentes)

    cols_finales = ["ID COB"] + vars_presentes + targets_presentes
    return df[cols_finales].reset_index(drop=True), reporte

## 3. Ejecutar sobre todas las hojas y exportar los `.parquet`



In [ ]:
xl = pd.ExcelFile(RUTA_EXCEL)
print(f"Pestañas encontradas: {xl.sheet_names}\n")

resumenes = []

for hoja in xl.sheet_names:
    df_limpio, reporte = cargar_y_limpiar_hoja(RUTA_EXCEL, hoja)

    if df_limpio is None:
        print(f"✗  {hoja}: OMITIDA — {reporte['errores']}")
        resumenes.append(reporte)
        continue

    ruta_parquet = CARPETA_SALIDA / f"{hoja}.parquet"
    df_limpio.to_parquet(ruta_parquet, index=False)

    print(f"✔  {hoja}: {reporte['n_filas_original']:,} filas originales -> "
          f"{reporte['n_filas_final']:,} finales | "
          f"{reporte['vars_presentes']} variables, {reporte['targets_presentes']} targets | "
          f"{len(reporte['advertencias'])} advertencias, {len(reporte['errores'])} errores")
    for a in reporte["advertencias"]:
        print(f"     ADVERTENCIA: {a}")
    for e in reporte["errores"]:
        print(f"     ERROR: {e}")
    print(f"     -> Guardado: {ruta_parquet.name}\n")

    resumenes.append(reporte)

## 4. Resumen de calidad de datos

Una tabla compacta para ver de un vistazo qué meses tuvieron más problemas, y decidir si algo amerita revisar el Excel de origen antes de entrenar.

In [ ]:
tabla_resumen = pd.DataFrame([
    {
        "Mes": r["hoja"],
        "Filas originales": r["n_filas_original"],
        "Filas finales": r["n_filas_final"],
        "Filas descartadas": r["n_filas_original"] - r["n_filas_final"],
        "Variables presentes": r.get("vars_presentes", 0),
        "Targets presentes": r.get("targets_presentes", 0),
        "# Advertencias": len(r["advertencias"]),
        "# Errores": len(r["errores"]),
    }
    for r in resumenes
])

tabla_resumen

## 5. Verificación final 

In [ ]:
for hoja in xl.sheet_names:
    ruta = CARPETA_SALIDA / f"{hoja}.parquet"
    if not ruta.exists():
        continue
    p = pd.read_parquet(ruta)
    ids_unicos = p["ID COB"].nunique() == len(p)

    targets_ok = True
    for t in TARGETS:
        if t in p.columns and not p[t].isin([0, 1]).all():
            targets_ok = False

    estado = "OK" if (ids_unicos and targets_ok) else "REVISAR"
    print(f"[{estado}] {hoja}: {p.shape[0]:,} filas x {p.shape[1]} cols | "
          f"IDs únicos: {ids_unicos} | Targets binarios: {targets_ok}")